# Agentic RAG with Function Calling

In [1]:
from openai import OpenAI
openai_client = OpenAI()

In [2]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules.\n\nIf you tell me:\n- the course name,\n- where it’s offered,\n- and whether it’s in progress or upcoming,\n\nI can help you figure out if you can still join and what to do next.'

#### Note: Ususally if the answer is generic, then its not from out KB

## Define the search Function that queries the index

In [3]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search('how do I run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

we are telling the LLM here how to do it and it is language agnostics - means its not language dependent, it can be written by JSON, python, java etc

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
# here we add the tools to be used for the search
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools = [search_tool]
)

In [8]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late join"}', call_id='call_FfbUC3DT9fJ1EGWKroEk04yU', name='search', type='function_call', id='fc_0a2fd7830cf5d172006a50a4cd53cc8198b9844d12a9e50c65', namespace=None, status='completed')]

In [9]:
len(response.output)


1

In [10]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late join"}', call_id='call_FfbUC3DT9fJ1EGWKroEk04yU', name='search', type='function_call', id='fc_0a2fd7830cf5d172006a50a4cd53cc8198b9844d12a9e50c65', namespace=None, status='completed')

In [11]:
call.arguments

'{"query":"Can I join the course if I just discovered it? enrollment eligibility late join"}'

In [12]:
import json

args = json.loads(call.arguments)

In [13]:
search(**args)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are grea

This is the argument that we need to send to the LLM

In [14]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are grea

#### Change results into json

In [15]:
result_json = json.dumps(results, indent =2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."
  },
  {
    "id": "193612db63",
    "course": "llm-zoomcamp",
    "section": "Module 3: Orchestration",
    "question": "Why do we need orchestration / Kestra \u2014 can't I just run

In [16]:
{
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

{'type': 'function_call_output',
 'call_id': 'call_FfbUC3DT9fJ1EGWKroEk04yU',
 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."\n  },\n  {\n    "id": "193612db63",\n    "course": "llm-zoomcamp",\n    "sect

#### Sending the history to the llm

In [17]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [18]:
#the original message we have
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late join"}', call_id='call_FfbUC3DT9fJ1EGWKroEk04yU', name='search', type='function_call', id='fc_0a2fd7830cf5d172006a50a4cd53cc8198b9844d12a9e50c65', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_FfbUC3DT9fJ1EGWKroEk04yU',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I e

The second call to the LLM

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

to calculate how much we spend on this LLM 

In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(979, 32)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


## Agentic Loop

We have a response from above now we make an agentic loop function 

In [28]:
def make_call(call):
    args = json.loads(call.arguments)
    if call.name == "search":
        results = search(**args)
    result_json = json.dumps(results, indent=2)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.


Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [43]:
response.output[0].content[0].text

'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

In [45]:
messages.extend(response.output)

for item in response.output:
    print(item)
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print('Assistant message:', item.content[0].text)
        print(item.content[0].text)

ResponseOutputMessage(id='msg_0a2fd7830cf5d172006a50a4cf26d0819887c7e4313587b9cd', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
Assistant message: Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.


In [33]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseOutputMessage(id='msg_0a2fd7830cf5d172006a50a4cf26d0819887c7e4313587b9cd', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

### wrapping the whole process in to one step and check for function calls, if there is no function call skip and give response.Keep track of iteration with i

In [46]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

i = 0
while True:
    print(f"--- Iteration {i} ---")
    has_function_call = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        print(item)
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_call = True

        elif item.type == "message":
            print('Assistant message:', item.content[0].text)
            print(item.content[0].text)

    i += 1
    if has_function_call == False:
        break

--- Iteration 0 ---
ResponseFunctionToolCall(arguments='{"query":"join course enroll discovered course can I join"}', call_id='call_FmkRT6zBgEEQYGCJEU2zloYZ', name='search', type='function_call', id='fc_03482661df01b126006a50aa64f5dc81998d1a3e6190694ace', namespace=None, status='completed')
function_call: search {"query":"join course enroll discovered course can I join"}
ResponseFunctionToolCall(arguments='{"query":"course enrollment eligibility late join discovered course"}', call_id='call_ghib00tm91kRI9K9iRaPI9cb', name='search', type='function_call', id='fc_03482661df01b126006a50aa64f5ec819985818c777524a8d8', namespace=None, status='completed')
function_call: search {"query":"course enrollment eligibility late join discovered course"}
--- Iteration 1 ---
ResponseOutputMessage(id='msg_03482661df01b126006a50aa6b2c9c8199a22a66d90390598c', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning and 

In [47]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search , analyze the results and then perform more searches based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [53]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

i = 0
while True:
    print(f"--- Iteration {i} ---")
    has_function_call = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        print(item)
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_call = True

        elif item.type == "message":
            print('Assistant message:', item.content[0].text)
            print(item.content[0].text)

    i += 1
    if has_function_call == False:
        break

--- Iteration 0 ---
ResponseFunctionToolCall(arguments='{"query":"join the course enroll registration can I join discovered course"}', call_id='call_6HNiN6IoJZub8ixPoQdyNjmP', name='search', type='function_call', id='fc_0093aeda62c7d780006a50b28bc238819bb0410dff7377d4dd', namespace=None, status='completed')
function_call: search {"query":"join the course enroll registration can I join discovered course"}
--- Iteration 1 ---
ResponseOutputMessage(id='msg_0093aeda62c7d780006a50b28d1e08819b9fdc57e4c59ef66c', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Otherwise, you can still start learning from the materials anytime.\n\nIf you want, I can also help you find the course docs, GitHub repo, or explain how the weekly workflow works.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
Assistant 

putting it into one function

In [54]:
def agentic_loop(question, instructions, model):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    i = 0
    while True:
        print(f"--- Iteration {i} ---")
        has_function_call = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            print(item)
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_call = True

            elif item.type == "message":
                print('Assistant message:', item.content[0].text)
                last_answer = item.content[0].text
                print(item.content[0].text)

        i += 1
        if has_function_call == False:
            break

    return last_answer

In [55]:
agentic_loop(question, instructions, model="gpt-5.4-mini")

--- Iteration 0 ---
ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join late enrollment FAQ"}', call_id='call_IGDAh3QtT07FteRhOEKZGJgk', name='search', type='function_call', id='fc_018f7d58d42c3730006a50b2be17e4819ba82347a3b5927e96', namespace=None, status='completed')
function_call: search {"query":"join the course discovered course can I join late enrollment FAQ"}
--- Iteration 1 ---
ResponseOutputMessage(id='msg_018f7d58d42c3730006a50b2c11964819b8713efff0c472456', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, the important thing is to submit your project while the course is still accepting submissions. Also, you don’t need a confirmation email to start learning or submitting homework while the form is open.\n\nIf you want, I can also help you figure out the best way to start the course now. Are there other areas you’d like to explore?', type='output_text', logprobs=[])], rol

'Yes — you can still join the course.\n\nIf you want a certificate, the important thing is to submit your project while the course is still accepting submissions. Also, you don’t need a confirmation email to start learning or submitting homework while the form is open.\n\nIf you want, I can also help you figure out the best way to start the course now. Are there other areas you’d like to explore?'

In [58]:
result = agentic_loop(question, instructions, model="gpt-5.4-mini")

--- Iteration 0 ---


ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered course can I join"}', call_id='call_lyLZXyC4bhMUtRRNpIFVhFmh', name='search', type='function_call', id='fc_0f418f30dcbdab5a006a50b3a1d1f481999c5e999880ea1cf5', namespace=None, status='completed')
function_call: search {"query":"join course enrollment discovered course can I join"}
--- Iteration 1 ---
ResponseFunctionToolCall(arguments='{"query":"certificate project while accepting submissions peer review live cohort self-paced join course discovered course"}', call_id='call_gsvtaXgIxyOPw9d6hmWKU9dZ', name='search', type='function_call', id='fc_0f418f30dcbdab5a006a50b3a2ba3881998523142337ce10a6', namespace=None, status='completed')
function_call: search {"query":"certificate project while accepting submissions peer review live cohort self-paced join course discovered course"}
--- Iteration 2 ---
ResponseOutputMessage(id='msg_0f418f30dcbdab5a006a50b3a418008199ac97c08ca0ee32e6', content=[ResponseOutputText(ann

In [59]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. The course can also be followed in a self-paced way, but certificates are only awarded for the live cohort.\n\nIf you’d like, I can also help with:\n- how to start the course,\n- whether you can still get a certificate,\n- or how the weekly workflow works.\n\nIs there anything else you want to explore?'

In [60]:
question = "What is the capital of France?"

In [62]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agentic_loop(question, instructions, model="gpt-5.4-mini")

--- Iteration 0 ---


ResponseFunctionToolCall(arguments='{"query":"capital of France"}', call_id='call_FzQJ497pTDk8rIDu7szY0Myq', name='search', type='function_call', id='fc_04d316d4c5f0e995006a50b5feae1c81988d2fc86fcd444cf9', namespace=None, status='completed')
function_call: search {"query":"capital of France"}
--- Iteration 1 ---
ResponseOutputMessage(id='msg_04d316d4c5f0e995006a50b601321481989343f5f827752953', content=[ResponseOutputText(annotations=[], text='I’m sorry, but I can’t help with off-topic questions like that.\n\nIf you have a course-related question, I’d be happy to help. Are there other areas you want to explore?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
Assistant message: I’m sorry, but I can’t help with off-topic questions like that.

If you have a course-related question, I’d be happy to help. Are there other areas you want to explore?
I’m sorry, but I can’t help with off-topic questions like that.

If you have a co

'I’m sorry, but I can’t help with off-topic questions like that.\n\nIf you have a course-related question, I’d be happy to help. Are there other areas you want to explore?'

## Using Datatalksclub locally developed function kit which is the same with agentic_loop above

In [63]:
!uv add toyaikit

Resolved 127 packages in 7.85s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/74.86 KiB           
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/74.86 KiB         
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/74.86 KiB         
httpx2               ------------------------------ 16.00 KiB/74.86 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/78.45 KiB           
httpx2               ------------------------------ 16.00 KiB/74.86 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.45 KiB         
httpx2               ------------------------------ 16.00 KiB/74.86 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.45 KiB         
httpx2               ------------------------------ 16.00 KiB/74.86 KiB
⠙ Preparing p

In [65]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [66]:
# defining the tools
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [67]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [68]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [69]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [70]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [71]:
#the actual agent
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [72]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [ ]:
result.cost

CostInfo(input_cost=Decimal('0.0034995'), output_cost=Decimal('0.0013005'), total_cost=Decimal('0.0048000'))

In [74]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install setup"}', call_id='call_rgSiwz

In [75]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run() 

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user